In [1]:
import numpy as np
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.model_selection import cross_val_score

In [2]:
X,y = fetch_california_housing(return_X_y=True)

# Using a subset (first 1000 rows) so that SVR runs quickly during our practice cross-validation
X,y = X[:,1000], y[:,1000]

IndexError: index 1000 is out of bounds for axis 1 with size 8

In [3]:
lr = LinearRegression()
dt = DecisionTreeRegressor(random_state=42)
svr = SVR()

estimators = [
    ('lr',lr),
    ('dt', dt),
    ('svr', svr)
]

for estimator in estimators:
    scores = cross_val_score(estimator[1],X,y,scoring='r2',cv=10)
    print(f"{estimator[0].upper():>3} R2 score: {np.round(np.mean(scores), 2)}")

 LR R2 score: 0.51
 DT R2 score: 0.24
SVR R2 score: -0.25


In [4]:
from sklearn.ensemble import VotingRegressor

vr = VotingRegressor(estimators)

scores = cross_val_score(vr, X,y, scoring='r2', cv=10)
print('Voting Regressor R2:', np.round(np.mean(scores),2))

Voting Regressor R2: 0.47


In [5]:
print("--- Weighted Voting Combinations ---")
best_score = -1
best_weights = None

# Testing weights from 1 to 3 for all three models
for i in range(1, 4):       # Weight for Linear Regression (LR)
    for j in range(1, 4):   # Weight for Decision Tree (DT)
        for k in range(1, 4):  # Weight for Support Vector Regressor (SVR)
            
            # Create Voting Regressor with current combination of weights
            vr_weighted = VotingRegressor(estimators, weights=[i, j, k])
            
            # Calculate cross-validated R2 score
            scores = cross_val_score(vr_weighted, X, y, scoring='r2', cv=10)
            avg_score = np.round(np.mean(scores), 2)
            
            # Track the best score
            if avg_score > best_score:
                best_score = avg_score
                best_weights = (i, j, k)
            
            print(f"Weights (LR={i}, DT={j}, SVR={k}) --> R2 Score: {avg_score}")

print(f"\nBest Weights: LR={best_weights[0]}, DT={best_weights[1]}, SVR={best_weights[2]}")
print(f"Best R2 Score: {best_score}")

--- Weighted Voting Combinations ---
Weights (LR=1, DT=1, SVR=1) --> R2 Score: 0.47
Weights (LR=1, DT=1, SVR=2) --> R2 Score: 0.37


KeyboardInterrupt: 

In [6]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import VotingRegressor
from sklearn.model_selection import cross_val_score
import numpy as np

# 1. Initialize 5 Decision Trees with different max_depths to create "Diversity"
dt1 = DecisionTreeRegressor(max_depth=1, random_state=42)
dt2 = DecisionTreeRegressor(max_depth=3, random_state=42)
dt3 = DecisionTreeRegressor(max_depth=5, random_state=42)
dt4 = DecisionTreeRegressor(max_depth=7, random_state=42)
dt5 = DecisionTreeRegressor(max_depth=None, random_state=42) # Fully grown tree

# 2. Store them in the estimators list
estimators_dt = [
    ('dt1', dt1), ('dt2', dt2), ('dt3', dt3), 
    ('dt4', dt4), ('dt5', dt5)
]

# 3. Check individual performance first
print("--- Individual Decision Tree Scores ---")
for estimator in estimators_dt:
    # Get cross-validated R2 score for each tree
    scores = cross_val_score(estimator[1], X, y, scoring='r2', cv=10)
    print(f"{estimator[0].upper()} (max_depth={estimator[1].max_depth}): {np.round(np.mean(scores), 2)}")

# 4. Combine all 5 trees using Voting Regressor
vr_same_algo = VotingRegressor(estimators_dt)
scores_vr = cross_val_score(vr_same_algo, X, y, scoring='r2', cv=10)

print("\n--- Voting Regressor (All 5 Combined) ---")
print(f"Combined R2 Score: {np.round(np.mean(scores_vr), 2)}")

--- Individual Decision Tree Scores ---
DT1 (max_depth=1): 0.13
DT2 (max_depth=3): 0.36
DT3 (max_depth=5): 0.43
DT4 (max_depth=7): 0.47
DT5 (max_depth=None): 0.24

--- Voting Regressor (All 5 Combined) ---
Combined R2 Score: 0.5
